# 课后练习解答（04.06_npu_postprocess_integration_offload）

本解答对应《NPU 后处理集成与异构卸载》课后练习，共 15 题。


### 问题1（单选题）

**题目：** 本节“卸载后处理”的含义最准确的是？

A. 把 YOLO NMS 从 CPU 侧迁移到 NPU 自定义算子执行
B. 把图片上传到 GitCode
C. 把 OM 转回 ONNX
D. 删除 CPU baseline

**解答：** A

**解析：** 实验五重点是将后处理 NMS 通过 ACLNN 自定义算子接入 NPU。


### 问题2（单选题）

**题目：** ACLNN runner 的输入文件主要来自哪里？

A. PyACL 得到的 YOLO OM 输出整理后的 boxes/scores
B. PPT 模板
C. Git 提交日志
D. NPU 温度表

**解答：** A

**解析：** 实验五从真实 OM 输出中整理候选框，再保存为 runner 输入。


### 问题3（单选题）

**题目：** 为什么真实流程中候选框数量可能变成 18866 一类的大数？

A. 真实 YOLO 输出过滤后候选框很多
B. NPU 自动复制图片
C. Git LFS 重复上传
D. CANN 把 batch 扩大为 18866

**解答：** A

**解析：** 真实 YOLO 输出有 25200 个候选位置，阈值过滤后仍可能留下大量候选框。


### 问题4（单选题）

**题目：** 如果 `YoloNmsAclNNInvocation/src/main.cpp` 中固定 shape 为 51，但实际 boxes 是 18866，会发生什么？

A. shape 不匹配，编译或运行验证流程会失败
B. 自动变快
C. 自动降为 CPU
D. OM 文件被删除

**解答：** A

**解析：** runner 的 tensor desc shape 必须和输入 bin/npz 数据规模一致。


### 问题5（多选题）

**题目：** NPU 后处理集成需要对齐哪些信息？

A. boxes shape
B. scores shape
C. keep/count 输出
D. CPU baseline 的 keep/count

**解答：** A、B、C、D

**解析：** 这些都是验证自定义 NMS 是否正确接入真实 YOLO 输出的关键。


### 问题6（多选题）

**题目：** 保存给 ACLNN runner 的输入文件时，通常需要保证什么？

A. float32 dtype
B. 内存连续
C. boxes 与 scores 顺序一致
D. 和 main.cpp 中 tensor desc shape 匹配

**解答：** A、B、C、D

**解析：** dtype、连续性、顺序和 shape 都会影响运行正确性。


### 问题7（多选题）

**题目：** 读取 NPU 输出后，常见的正确性检查包括哪些？

A. custom count 是否等于 CPU count
B. custom keep 是否等于 CPU keep
C. 映射回 model row indices 是否一致
D. 只看 runner 返回码是否为 0

**解答：** A、B、C

**解析：** 返回码为 0 不等于算法结果正确，还要比较输出。


### 问题8（判断题）

**题目：** 实验五已经不再使用 OM 输出，只是在跑随机小样例。

**解答：** 错误

**解析：** 实验五将 PyACL/OM 输出整理成真实 NMS 输入，比最初小样例更接近完整部署流程。


### 问题9（判断题）

**题目：** Python wrapper `import yolo_nms_custom` 是本实验后续验证的必要条件。

**解答：** 错误

**解析：** 当前流程通过 ACLNN C++ runner 调用自定义算子，不依赖 Python 封装。


### 问题10（填空题）

**题目：** 本节保存 ACLNN runner 输入的中间文件通常是 `outputs/____`。

**解答：** yolo_nms_inputs.npz

**解析：** 该 npz 保存 boxes、scores、CPU 参考 keep/count 等信息。


### 问题11（填空题）

**题目：** NPU NMS 输出中，`count` 表示 `____`。

**解答：** 有效保留框数量

**解析：** keep 数组可能预留 max_output 长度，count 指明前多少个元素有效。


### 问题12（简答题）

**题目：** 实验四的小样例验证和实验五真实 YOLO 输出验证是什么关系？

**解答：** 实验四先用较小、可控的 boxes/scores 验证自定义算子工程、编译、安装和 ACLNN 调用链路；实验五再把真实 OM 输出接入同一个 NMS 算子，验证它能处理真实后处理输入。二者互通但验证层次不同。

**解析：** 一个验证算子最小闭环，一个验证与 YOLO 部署链路集成。


### 问题13（简答题）

**题目：** 为什么实验五需要把候选框先按 score 降序排序？

**解答：** 当前 Ascend C kernel 是最小验证实现，假设输入已经按 score 从高到低排列，因此 Python 侧先排序，kernel 内只负责 IoU 抑制和写出 keep/count。

**解析：** 这是功能优先的实现策略，后续可把排序也下沉到 NPU。


### 问题14（简答题）

**题目：** runner 返回码为 0 但没有 PASSED，应该怎么判断问题？

**解答：** 返回码为 0 只说明程序执行没有崩溃。应继续检查 verify_result.py 的输出，比较 custom keep/count 与 CPU golden 是否一致，确认 shape、输入顺序和 dtype 是否正确。

**解析：** 正确性必须看输出对齐，而不是只看进程成功。


### 问题15（代码设计题）

**题目：** 写一段 Python 代码，根据 boxes 数量生成 main.cpp 中需要的 shape 数值。

**解答：**

```python
import numpy as np
pack = np.load("outputs/yolo_nms_inputs.npz")
boxes = pack["boxes"].astype(np.float32)
scores = pack["scores"].astype(np.float32)
print("boxesShape =", "{" + f"{boxes.shape[0]}, 4" + "}")
print("scoresShape =", "{" + f"{scores.shape[0]}" + "}")
print("keepShape = {300}")
print("countShape = {1}")
```

**解析：** 得到的 N 必须同步写入 YoloNmsAclNNInvocation/src/main.cpp 后重新编译运行。
